# 💼 Agent Job Tracker

An intelligent pipeline that monitors your email for job application updates, infers the current status of each application using LLMs (Google Gemini), and drafts context-aware follow-up emails based on the time elapsed since the last contact.

### Key Features:
- **Privacy First**: Local keyword-based classification ensures only job-related threads are processed by the LLM.
- **LLM Powered**: Uses Gemini for high-accuracy status inference and personalized drafting.
- **Graceful Degradation**: Falls back to deterministic rule-based logic if API keys are missing or quotas are reached.
- **Business-Day Aware**: Calculates follow-up urgency using a calendar that ignores weekends and US holidays.

## 1. Setup & Installation
Install the required dependencies.

In [1]:
!pip install google-generativeai python-dotenv

## 2. Configuration
This section handles the configuration. You can either set up a `.env` file or manually fill in the `Config` object below.

In [2]:
import os
from dataclasses import dataclass
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

@dataclass(frozen=True)
class Config:
    imap_host: str = "imap.gmail.com"
    imap_port: int = 993
    imap_username: str = os.getenv("IMAP_USERNAME", "")
    imap_password: str = os.getenv("IMAP_PASSWORD", "")
    imap_folder: str = "INBOX"
    imap_days_back: int = 60
    gemini_api_key: str | None = os.getenv("GEMINI_API_KEY")
    gemini_model: str = os.getenv("GEMINI_MODEL", "gemini-1.5-flash")
    monitoring_alert_days: int = 10
    confidence_threshold: float = 0.6

config = Config()

## 3. Core Modules
The following cells contain the logic previously split into separate files.

In [3]:
import imaplib
import re
import ssl
import json
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone, date
from email import policy
from email.message import EmailMessage
from email.parser import BytesParser
from email.utils import getaddresses, parsedate_to_datetime
from typing import Any

# --- Email Fetcher ---
@dataclass
class EmailMessage_:
    message_id: str
    from_: str
    to: str
    subject: str
    date: datetime
    body: str
    in_reply_to: str | None = None
    references: str | None = None

@dataclass
class Thread_:
    thread_id: str
    subject: str
    messages: list[EmailMessage_] = field(default_factory=list)
    participants: set[str] = field(default_factory=set)
    last_date: datetime | None = None

    @property
    def last_contact(self) -> datetime | None:
        return self.last_date

    def as_text_corpus(self, max_chars: int = 12_000) -> str:
        ordered = sorted(self.messages, key=lambda m: m.date)
        chunks: list[str] = []
        total = 0
        for m in ordered:
            chunk = (f"From: {m.from_}\nTo: {m.to}\nDate: {m.date.isoformat()}\n"
                    f"Subject: {m.subject}\n\n{m.body.strip()}\n{'-' * 40}\n")
            if total + len(chunk) > max_chars:
                chunks.append("[…earlier messages truncated to fit context window]")
                break
            chunks.append(chunk)
            total += len(chunk)
        return "\n".join(chunks)

def _strip_subject(subject: str) -> str:
    return re.sub(r"^\s*(re|fwd|fw)\s*:\s*", "", subject.strip(), flags=re.IGNORECASE).strip()

def _thread_key(msg: EmailMessage_) -> str:
    if msg.references:
        first = msg.references.split()[0].strip("<>")
        if first: return f"ref:{first}"
    if msg.in_reply_to:
        return f"irt:{msg.in_reply_to.strip('<>')}"
    return f"subj:{_strip_subject(msg.subject).lower()[:120]}"

def consolidate_threads(messages: list[EmailMessage_]) -> list[Thread_]:
    by_key: dict[str, Thread_] = {}
    for msg in messages:
        key = _thread_key(msg)
        t = by_key.get(key)
        if t is None:
            t = Thread_(thread_id=key, subject=_strip_subject(msg.subject))
            by_key[key] = t
        t.messages.append(msg)
        t.participants.update(a for _, a in getaddresses([msg.from_, msg.to]) if a)
        if t.last_date is None or msg.date > t.last_date:
            t.last_date = msg.date
    return sorted(by_key.values(), key=lambda t: t.last_date or datetime.min, reverse=True)

def fetch_threads(cfg: Config) -> list[Thread_]:
    if not cfg.imap_username or not cfg.imap_password:
        raise RuntimeError("IMAP credentials not set.")
    ctx = ssl.create_default_context()
    with imaplib.IMAP4_SSL(cfg.imap_host, cfg.imap_port, ssl_context=ctx) as imap:
        imap.login(cfg.imap_username, cfg.imap_password)
        imap.select(cfg.imap_folder)
        since = (datetime.now(timezone.utc) - timedelta(days=cfg.imap_days_back)).strftime("%d-%b-%Y")
        typ, data = imap.search(None, f'(SINCE {since})')
        if typ != "OK" or not data or not data[0]: return []
        ids = data[0].split()
        out = []
        for msg_id in ids:
            typ, raw = imap.fetch(msg_id, "(RFC822)")
            if typ != "OK" or not raw or not raw[0]: continue
            parsed = BytesParser(policy=policy.default).parsebytes(raw[0][1])
            out.append(_parse_message(parsed))
        return consolidate_threads(out)

def _parse_message(msg: EmailMessage) -> EmailMessage_:
    body = _extract_body(msg)
    date_hdr = msg.get("Date", "")
    try:
        date = parsedate_to_datetime(date_hdr)
        if date.tzinfo is None: date = date.replace(tzinfo=timezone.utc)
    except Exception: date = datetime.now(timezone.utc)
    return EmailMessage_(
        message_id=(msg.get("Message-ID") or "").strip(),
        from_=msg.get("From", ""),
        to=msg.get("To", ""),
        subject=msg.get("Subject", ""),
        date=date, body=body, in_reply_to=msg.get("In-Reply-To"), references=msg.get("References")
    )

def _extract_body(msg: EmailMessage) -> str:
    if msg.is_multipart():
        text_parts, html_parts = [], []
        for part in msg.walk():
            ctype = part.get_content_type()
            if "attachment" in (part.get("Content-Disposition") or "").lower(): continue
            payload = part.get_content()
            if payload is None: continue
            if ctype == "text/plain": text_parts.append(str(payload))
            elif ctype == "text/html": html_parts.append(str(payload))
        if text_parts: return "\n".join(text_parts)
        if html_parts:
            stripped = re.sub(r"<[^>]+>", " ", html_parts[0])
            return re.sub(r"\s+", " ", stripped).strip()
        return ""
    payload = msg.get_content()
    return str(payload) if payload is not None else ""

# --- Thread Classifier ---
_JOB_KEYWORDS = re.compile(r"\b(application|applied|applicant|resume|cv|cover letter|interview|phone screen|hiring|recruiter|talent acquisition|job offer|offer letter|position|role|opening|requisition|job description|jd|on[- ]site|onsite|thank you for (your|applying)|unfortunately|we have decided|moving forward|take[- ]home|assessment|coding challenge|reference check|background check|offer(?:ed)?|compensation|salary|start date)\b", re.IGNORECASE)
_NON_JOB_HINTS = re.compile(r"\b(invoice|receipt|payment due|subscription|order confirmation|reset your password|verify your account|weekly digest|newsletter|promo|sale|coupon|reservation|itinerary|confirmation number)\b", re.IGNORECASE)

@dataclass
class Classification:
    is_job_thread: bool
    reason: str
    company_hint: str | None = None
    role_hint: str | None = None

def classify(thread: Thread_) -> Classification:
    corpus = thread.as_text_corpus(max_chars=20_000)
    subject = thread.subject or ""
    if _NON_JOB_HINTS.search(corpus) and not _JOB_KEYWORDS.search(corpus):
        return Classification(False, "matched non-job hints only")
    if _JOB_KEYWORDS.search(corpus) or _JOB_KEYWORDS.search(subject):
        return Classification(True, "matched job keywords", _guess_company(thread), _guess_role(thread))
    if re.search(r"\b(application|role|position|interview)\b", subject, re.IGNORECASE):
        return Classification(True, "subject hints at application", _guess_company(thread), _guess_role(thread))
    return Classification(False, "no job indicators")

def _guess_company(thread: Thread_) -> str | None:
    for m in thread.messages:
        for _, addr in getaddresses([m.from_]):
            if "@" not in addr: continue
            domain = addr.split("@", 1)[1].lower().removeprefix("mail.").removeprefix("noreply.").removeprefix("www.")
            head = domain.split(".", 1)[0]
            if head and head not in {"gmail", "outlook", "yahoo", "icloud", "hotmail"}: return head.title()
    if thread.subject:
        first = thread.subject.split(" - ")[0].strip()
        if 1 < len(first) <= 30 and first.lower() not in {"re", "fwd", "fw"}: return first
    return None

def _guess_role(thread: Thread_) -> str | None:
    s = thread.subject
    m = re.search(r"[-–—]\s*([A-Z][A-Za-z/ &.+#-]{3,60}?)\s*[-–—]", s)
    if m: return m.group(1).strip()
    m = re.search(r"\b((?:Senior|Junior|Staff|Principal)?\s?[A-Za-z]+(?:\s+[A-Za-z]+){0,3})\s+(?:role|position|application|interview)\b", s, re.IGNORECASE)
    if m: return m.group(1).strip()
    return None

# --- Gemini Helpers ---
def _pick_working_model(genai: Any, configured: str) -> str:
    try:
        genai.GenerativeModel(model_name=configured).count_tokens("ping")
        return configured
    except Exception:
        try:
            for m in genai.list_models():
                if "generateContent" in getattr(m, "supported_generation_methods", []):
                    name = m.name.split("/")[-1]
                    if name.startswith("gemini"): return name
        except Exception: pass
    return configured

def is_quota_error(exc: BaseException) -> bool:
    try:
        from google.api_core import exceptions as gexc
        return isinstance(exc, (gexc.ResourceExhausted, gexc.TooManyRequests))
    except ImportError: return False

# --- Status Inference ---
VALID_STATUSES = ("applied", "interview_scheduled", "rejected", "offer", "no_response", "unclear")

@dataclass
class StatusInference:
    status: str
    confidence: float
    signals: list[str]
    last_role: str | None = None
    last_company: str | None = None
    raw: dict[str, Any] | None = None

def infer_status(thread: Thread_, cfg: Config) -> StatusInference:
    if not cfg.gemini_api_key:
        return _rule_based_status(thread, "no API key")
    try:
        import google.generativeai as genai
        genai.configure(api_key=cfg.gemini_api_key)
        prompt = (f"You classify a single email thread about a job application.\n"
                  f"Choose exactly one status from: {VALID_STATUSES}. Confidence [0,1]. "
                  f"Signals are quoted phrases. If unsure, status='unclear' and confidence<0.5.\n\n"
                  f"SUBJECT: {thread.subject}\nPARTICIPANTS: {', '.join(sorted(thread.participants))}\n\n"
                  f"THREAD CORPUS:\n{thread.as_text_corpus()}")
        model_name = _pick_working_model(genai, cfg.gemini_model)
        model = genai.GenerativeModel(
            model_name=model_name, 
            generation_config={"response_mime_type": "application/json", "temperature": 0.2}
        )
        resp = model.generate_content(prompt)
        data = json.loads(resp.text)
        status = str(data.get("status", "unclear")).lower()
        if status not in VALID_STATUSES: status = "unclear"
        conf = float(data.get("confidence", 0.0))
        if conf < cfg.confidence_threshold and status != "unclear": status = "unclear"
        return StatusInference(status, conf, list(data.get("signals", []) or []), data.get("role"), data.get("company"), data)
    except Exception as e:
        if is_quota_error(e): return _rule_based_status(thread, "quota hit")
        return _rule_based_status(thread, f"error: {e}")

def _rule_based_status(thread: Thread_, reason: str) -> StatusInference:
    lower = thread.as_text_corpus(max_chars=8_000).lower()
    if re.search(r"\b(offer|offer letter|pleased to extend)\b", lower): s, c, sig = "offer", 0.9, ["offer language"]
    elif re.search(r"\b(unfortunately|moving forward with other|we have decided|rejected)\b", lower): s, c, sig = "rejected", 0.85, ["rejection language"]
    elif re.search(r"\b(interview|phone screen|schedule|onsite|panel)\b", lower): s, c, sig = "interview_scheduled", 0.8, ["interview language"]
    elif re.search(r"\b(received your application|thanks for applying)\b", lower): s, c, sig = "applied", 0.75, ["app acknowledgement"]
    else: s, c, sig = "no_response", 0.5, ["no signal"]
    return StatusInference(s, c, sig + [reason])

# --- Business Days ---
def _parse_date(d: str | datetime | date) -> date:
    if isinstance(d, datetime): return d.date()
    if isinstance(d, date): return d
    return datetime.fromisoformat(d).date()

def between(start, end, holidays=None) -> int:
    s, e = _parse_date(start), _parse_date(end)
    if e <= s: return 0
    if holidays is None: holidays = {date(s.year, 1, 1), date(s.year, 7, 4), date(s.year, 12, 25)}
    days, cur = 0, s + timedelta(days=1)
    while cur <= e:
        if cur.weekday() < 5 and cur not in holidays: days += 1
        cur += timedelta(days=1)
    return days

def tone_bucket(days: int) -> str:
    if days <= 14: return "warm"
    if days <= 30: return "firm"
    return "closing"

# --- Follow Up Drafter ---
@dataclass
class FollowUpDraft:
    subject: str
    body: str
    tone: str
    status_at_draft: str

def draft_follow_up(thread: Thread_, status: StatusInference, cfg: Config, bdays: int) -> FollowUpDraft | None:
    if status.status in {"rejected", "offer"}: return None
    tone = tone_bucket(bdays)
    if not cfg.gemini_api_key:
        return _rule_based_draft(thread, status, tone)
    try:
        import google.generativeai as genai
        genai.configure(api_key=cfg.gemini_api_key)
        prompt = (f"Draft a follow-up email. Ground it in the thread. Tone: {tone}. "
                  f"Role: {status.last_role}, Company: {status.last_company}. "
                  f"Status: {status.status}, Days since contact: {bdays}.\n\n"
                  f"THREAD:\n{thread.as_text_corpus()}")
        model = genai.GenerativeModel("gemini-1.5-flash", generation_config={"response_mime_type": "application/json"})
        resp = model.generate_content(prompt)
        data = json.loads(resp.text)
        return FollowUpDraft(data.get("subject", ""), data.get("body", ""), tone, status.status)
    except Exception:
        return _rule_based_draft(thread, status, tone)

def _rule_based_draft(thread, status, tone):
    role, comp = status.last_role or "the role", status.last_company or "your team"
    subj = f"Following up — {role} at {comp}"
    if tone == "warm": body = f"Hi team,\n\nI wanted to quickly follow up on my application for the {role} role. Happy to share anything else that would help.\n\nThanks!"
    elif tone == "firm": body = f"Hi team,\n\nChecking in on the {role} role. Could you let me know where things stand?\n\nThanks!"
    else: body = f"Hi team,\n\nI haven't heard back on the {role} role. Should I assume the file is closed?\n\nThanks!"
    return FollowUpDraft(subj, body, tone, status.status)

# --- Pipeline ---
@dataclass
class ApplicationRecord:
    thread_id: str
    subject: str
    company: str | None
    role: str | None
    status: str
    confidence: float
    signals: list[str]
    last_contact: str
    business_days_elapsed: int
    needs_follow_up: bool
    draft_subject: str | None = None
    draft_body: str | None = None
    draft_tone: str | None = None

def process_threads(threads: list[Thread_], cfg: Config) -> list[ApplicationRecord]:
    now = datetime.now(timezone.utc)
    results = []
    for thread in threads:
        cls = classify(thread)
        if not cls.is_job_thread: continue
        status = infer_status(thread, cfg)
        company = status.last_company or cls.company_hint
        role = status.last_role or cls.role_hint
        last = thread.last_contact or now
        bdays = between(last, now)
        needs = bdays >= cfg.monitoring_alert_days
        draft = draft_follow_up(thread, status, cfg, bdays) if needs else None
        results.append(ApplicationRecord(
            thread.thread_id, thread.subject, company, role, status.status, 
            status.confidence, status.signals, last.isoformat(), bdays, needs, 
            draft.subject if draft else None, draft.body if draft else None, draft.tone if draft else None
        ))
    return sorted(results, key=lambda a: a.last_contact, reverse=True)

def print_digest(apps: list[ApplicationRecord]):
    print(f"\n=== Job Tracker Digest ===")
    for app in apps:
        flag = " !! follow-up" if app.needs_follow_up else ""
        print(f"\n* {app.company or '?'} - {app.role or '?'} {flag}")
        print(f"  status: {app.status} (conf={app.confidence:.2f})")
        print(f"  last contact: {app.last_contact} ({app.business_days_elapsed} bd)")
        if app.draft_subject:
            print(f"  draft: {app.draft_subject}\n    {app.draft_body[:100]}...")

## 4. Run the Pipeline
You can run this with sample data to see it in action without needing API keys.

In [4]:
def load_sample_data():
    now = datetime(2026, 8, 28, tzinfo=timezone.utc)
    msgs = [
        EmailMessage_("a1", "careers@acme.com", "me@me.com", "Acme - Senior Backend - Offer", now - timedelta(days=2), "We are thrilled to extend an offer!"),
        EmailMessage_("b1", "no-reply@beta.io", "me@me.com", "Re: BetaLabs - Decision", now - timedelta(days=5), "We have decided to move forward with another candidate."),
        EmailMessage_("c1", "recruiter@cloud.com", "me@me.com", "CloudOps - Phone screen", now - timedelta(days=18), "Would love to set up a call!"),
        EmailMessage_("d1", "careers@data.com", "me@me.com", "Thanks for applying", now - timedelta(days=22), "Received your application. We will be in touch."),
    ]
    return consolidate_threads(msgs)

print("Running with sample data...")
sample_threads = load_sample_data()
results = process_threads(sample_threads, config)
print_digest(results)

Running with sample data...

=== Job Tracker Digest ===

* Acme - Senior Backend 
  status: offer (conf=0.90)
  last contact: 2026-08-26T00:00:00+00:00 (8 bd)

* Beta - ?  !! follow-up
  status: rejected (conf=0.85)
  last contact: 2026-08-23T00:00:00+00:00 (11 bd)

* Cloud - ?  !! follow-up
  status: interview_scheduled (conf=0.80)
  last contact: 2026-08-10T00:00:00+00:00 (20 bd)
  draft: Following up — the role at your team
    Hi team,

Checking in on the the role role. Could you let me know where things stand?

Thanks!...

* Data - ?  !! follow-up
  status: applied (conf=0.75)
  last contact: 2026-08-06T00:00:00+00:00 (22 bd)
  draft: Following up — the role at your team
    Hi team,

Checking in on the the role role. Could you let me know where things stand?

Thanks!...


## 5. Live Run (Optional)
Uncomment the code below and ensure your IMAP and Gemini credentials are set in the config to process your real emails.

In [6]:
try:
    live_threads = fetch_threads(config)
    live_results = process_threads(live_threads, config)
    print_digest(live_results)
except Exception as e:
    print(f"Live run failed: {e}")

Live run failed: IMAP credentials not set.
